# 01 — Paper-faithful VGG16 reproduction

This notebook is an entry point to the reusable source code. It does not train unless you explicitly set `RUN_TRAINING = True`. The baseline uses grayscale repeated across RGB channels, direct resize, `/255` scaling, a frozen ImageNet VGG16 base, and no augmentation/class weighting/fine-tuning.

## Colab or local setup

In Colab, clone or upload this repository and optionally mount Drive. Keep dataset and run-output folders outside the repository. Set `PROJECT_ROOT`, `BUSI_ROOT`, and `PROJECT_OUTPUT_ROOT` as needed. Install dependencies from `requirements.txt` in the runtime.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# In Colab, set this to the path where you cloned/uploaded the repository.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/Breast-Cancer-Detection-VGG16"))
if not (PROJECT_ROOT / "src" / "train.py").is_file():
    raise FileNotFoundError(f"Set PROJECT_ROOT to the repository folder. Current value: {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))

# Optional Drive integration. Leave False to use Colab runtime storage.
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

_DEFAULT_DATA = "/content/drive/MyDrive/BreastCancerProject/BUSI" if USE_GOOGLE_DRIVE else "/content/data/BUSI"
_DEFAULT_OUTPUT = "/content/drive/MyDrive/BreastCancerProject/reproducible_runs" if USE_GOOGLE_DRIVE else "/content/busi_runs"
BUSI_ROOT = Path(os.environ.get("BUSI_ROOT", _DEFAULT_DATA))
OUTPUT_ROOT = Path(os.environ.get("PROJECT_OUTPUT_ROOT", _DEFAULT_OUTPUT))
RUN_TRAINING = False  # Change to True only when you intend to train this experiment.


## Inspect dataset and planned split (no training)

In [ ]:
from src.preprocessing import discover_busi_images, make_stratified_splits

records = discover_busi_images(BUSI_ROOT)
splits = make_stratified_splits(records, seed=42)
print("Total images:", len(records))
print("Train:", len(splits.train), "Validation:", len(splits.validation), "Test:", len(splits.test))
print("Split unit: image-level; related images may cross splits.")


## Optional training

Training can take time and may download ImageNet weights. The command below defaults to disabled.

In [ ]:
if RUN_TRAINING:
    command = [
        sys.executable, "-m", "src.train",
        "--experiment", "baseline",
        "--data-root", str(BUSI_ROOT),
        "--output-root", str(OUTPUT_ROOT),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Training is disabled. Set RUN_TRAINING = True after reviewing the configuration.")
